# Create 6000-Episode Final-Step Dev Subset

This notebook creates a smaller final-step dev split from the 9000-episode split.

Policy:

- keep every episode whose evaluator gold label is a positive relation
- randomly sample episode-level `no_relation` cases with a fixed seed
- target exactly 6000 episodes

The evaluator gold label is positive when the query relation appears among the episode support-way relations. Otherwise the episode is `no_relation`, even if the raw query relation is not literally `no_relation`.


In [1]:
from pathlib import Path
from collections import Counter
import json
import pickle
import random


In [2]:
DATA_DIR = Path("../data")
DATASET_TYPE = "fs_tacred"
SOURCE_SPLIT = "final_step_dev"
OUTPUT_SPLIT = "final_step_dev_6000"
TARGET_EPISODES = 6000
SEED = 170000
QUERY_INDEX = 0
NO_RELATION = "no_relation"

SOURCE_EPISODES_PATH = DATA_DIR / f"{DATASET_TYPE}_{SOURCE_SPLIT}_episodes_1shots.pkl"
SOURCE_DETAILS_PATH = DATA_DIR / f"{DATASET_TYPE}_{SOURCE_SPLIT}_episodes_shots_details.pkl"
OUTPUT_EPISODES_PATH = DATA_DIR / f"{DATASET_TYPE}_{OUTPUT_SPLIT}_episodes_1shots.pkl"
OUTPUT_DETAILS_PATH = DATA_DIR / f"{DATASET_TYPE}_{OUTPUT_SPLIT}_episodes_shots_details.pkl"
OUTPUT_METADATA_PATH = DATA_DIR / f"{DATASET_TYPE}_{OUTPUT_SPLIT}_metadata.json"

print("source episodes:", SOURCE_EPISODES_PATH)
print("source details:", SOURCE_DETAILS_PATH)
print("output episodes:", OUTPUT_EPISODES_PATH)
print("output details:", OUTPUT_DETAILS_PATH)
print("output metadata:", OUTPUT_METADATA_PATH)


source episodes: ../data/fs_tacred_final_step_dev_episodes_1shots.pkl
source details: ../data/fs_tacred_final_step_dev_episodes_shots_details.pkl
output episodes: ../data/fs_tacred_final_step_dev_6000_episodes_1shots.pkl
output details: ../data/fs_tacred_final_step_dev_6000_episodes_shots_details.pkl
output metadata: ../data/fs_tacred_final_step_dev_6000_metadata.json


In [3]:
with SOURCE_EPISODES_PATH.open("rb") as f:
    episodes = pickle.load(f)

with SOURCE_DETAILS_PATH.open("rb") as f:
    details = pickle.load(f)

shots = details["shots"]
queries = details["queries"]
umbc_shots = details.get("umbc_shots", {})

print("episodes:", len(episodes))
print("shots:", len(shots))
print("queries:", len(queries))
print("umbc_shots:", len(umbc_shots))

assert len(episodes) == 9000
assert TARGET_EPISODES <= len(episodes)


episodes: 9000
shots: 633
queries: 7365
umbc_shots: 0


In [6]:
def resolve_way_relation(way, shots):
    # In these 1-shot files each way is usually a one-element list containing a shot id.
    if isinstance(way, list):
        shot_id = way[0]
    else:
        shot_id = way
    return shots[shot_id]["relation"]


def episode_gold_label(episode, shots, queries, query_index=0):
    query_id = episode["meta_test"][query_index]
    query_relation = queries[query_id]["relation"]
    way_relations = [resolve_way_relation(way, shots) for way in episode["meta_train"]]
    if query_relation in way_relations:
        return query_relation
    return NO_RELATION


labels = [episode_gold_label(ep, shots, queries, QUERY_INDEX) for ep in episodes]
label_counts = Counter(labels)

print("label counts:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

positive_indices = [i for i, label in enumerate(labels) if label != NO_RELATION]
negative_indices = [i for i, label in enumerate(labels) if label == NO_RELATION]

print("positive episodes:", len(positive_indices))
print("negative episodes:", len(negative_indices))

assert len(positive_indices) > 0
assert len(positive_indices) < TARGET_EPISODES
assert len(negative_indices) >= TARGET_EPISODES - len(positive_indices)


label counts:
  no_relation: 8801
  per:age: 77
  org:country_of_headquarters: 61
  org:parents: 29
  org:founded: 11
  per:stateorprovince_of_death: 11
  per:alternate_names: 10
positive episodes: 199
negative episodes: 8801


In [9]:
rels = set()
cnta = 0
cntn = 0
for v in queries.values():
    rels.add(v["relation"])
    cnta += 1
    if v["relation"] == "no_relation":
        cntn += 1

print(cntn)
print(cnta)

print(len(rels))
# for rr in rels:
#     print(rr)


5606
7365
42


In [10]:
rng = random.Random(SEED)
n_negative_needed = TARGET_EPISODES - len(positive_indices)
selected_negative_indices = rng.sample(negative_indices, n_negative_needed)
selected_indices = sorted(positive_indices + selected_negative_indices)
subset_episodes = [episodes[i] for i in selected_indices]
subset_labels = [labels[i] for i in selected_indices]

print("selected episodes:", len(subset_episodes))
print("selected positives:", sum(label != NO_RELATION for label in subset_labels))
print("selected negatives:", sum(label == NO_RELATION for label in subset_labels))
print("selected label counts:")
for label, count in Counter(subset_labels).most_common():
    print(f"  {label}: {count}")

assert len(subset_episodes) == TARGET_EPISODES
assert set(positive_indices).issubset(selected_indices)
assert sum(label != NO_RELATION for label in subset_labels) == len(positive_indices)


selected episodes: 6000
selected positives: 199
selected negatives: 5801
selected label counts:
  no_relation: 5801
  per:age: 77
  org:country_of_headquarters: 61
  org:parents: 29
  org:founded: 11
  per:stateorprovince_of_death: 11
  per:alternate_names: 10


In [11]:
# Keep only details referenced by the subset to make the output details file smaller and self-contained.
selected_shot_ids = set()
selected_query_ids = set()

for episode in subset_episodes:
    selected_query_ids.update(episode["meta_test"])
    for way in episode["meta_train"]:
        if isinstance(way, list):
            selected_shot_ids.update(way)
        else:
            selected_shot_ids.add(way)

subset_details = {
    "shots": {shot_id: shots[shot_id] for shot_id in selected_shot_ids},
    "queries": {query_id: queries[query_id] for query_id in selected_query_ids},
    "umbc_shots": umbc_shots,
}

print("subset shots:", len(subset_details["shots"]))
print("subset queries:", len(subset_details["queries"]))
print("subset umbc_shots:", len(subset_details["umbc_shots"]))


subset shots: 633
subset queries: 5265
subset umbc_shots: 0


In [12]:
metadata = {
    "dataset_type": DATASET_TYPE,
    "source_split": SOURCE_SPLIT,
    "output_split": OUTPUT_SPLIT,
    "source_episodes_path": str(SOURCE_EPISODES_PATH),
    "source_details_path": str(SOURCE_DETAILS_PATH),
    "output_episodes_path": str(OUTPUT_EPISODES_PATH),
    "output_details_path": str(OUTPUT_DETAILS_PATH),
    "target_episodes": TARGET_EPISODES,
    "seed": SEED,
    "query_index": QUERY_INDEX,
    "source_episode_count": len(episodes),
    "selected_episode_count": len(subset_episodes),
    "positive_episode_count": len(positive_indices),
    "selected_negative_episode_count": n_negative_needed,
    "source_label_counts": dict(label_counts),
    "selected_label_counts": dict(Counter(subset_labels)),
    "selected_indices": selected_indices,
}

with OUTPUT_EPISODES_PATH.open("wb") as f:
    pickle.dump(subset_episodes, f)

with OUTPUT_DETAILS_PATH.open("wb") as f:
    pickle.dump(subset_details, f)

with OUTPUT_METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("wrote:", OUTPUT_EPISODES_PATH)
print("wrote:", OUTPUT_DETAILS_PATH)
print("wrote:", OUTPUT_METADATA_PATH)


wrote: ../data/fs_tacred_final_step_dev_6000_episodes_1shots.pkl
wrote: ../data/fs_tacred_final_step_dev_6000_episodes_shots_details.pkl
wrote: ../data/fs_tacred_final_step_dev_6000_metadata.json


In [13]:
# Reload and verify output files.
with OUTPUT_EPISODES_PATH.open("rb") as f:
    reloaded_episodes = pickle.load(f)

with OUTPUT_DETAILS_PATH.open("rb") as f:
    reloaded_details = pickle.load(f)

reloaded_labels = [
    episode_gold_label(ep, reloaded_details["shots"], reloaded_details["queries"], QUERY_INDEX)
    for ep in reloaded_episodes
]

print("reloaded episodes:", len(reloaded_episodes))
print("reloaded label counts:", Counter(reloaded_labels))
print("reloaded shots:", len(reloaded_details["shots"]))
print("reloaded queries:", len(reloaded_details["queries"]))

assert len(reloaded_episodes) == TARGET_EPISODES
assert Counter(reloaded_labels) == Counter(subset_labels)


reloaded episodes: 6000
reloaded label counts: Counter({'no_relation': 5801, 'per:age': 77, 'org:country_of_headquarters': 61, 'org:parents': 29, 'org:founded': 11, 'per:stateorprovince_of_death': 11, 'per:alternate_names': 10})
reloaded shots: 633
reloaded queries: 5265
